# Etapa 3 - MapReduce con Spark RDDs

En este notebook resolvemos cuatro preguntas de negocio del dominio de la fabrica de pastas usando la API de RDDs de PySpark. En cada caso se detalla que ocurre en la fase de **Map**, que sucede en la fase de **Reduce** y cuando Spark ejecuta efectivamente el computo.

Los datos utilizados provienen de los archivos CSV generados a partir de la Etapa 1.

## Consultas analizadas

1. ¿Qué ingredientes aparecen en más rellenos distintos?
2. ¿Cuál es el gasto promedio por compra en cada franquicia?
3. ¿Cuáles son las 10 pastas más vendidas por kilos?
4. ¿Qué meses concentran la mayor cantidad de compras?


## Requisitos

- Tener instalado `pyspark`.
- Tener Java 11 o 17 disponible en el sistema.
- Tener Python 3.10+ instalado.
- Ejecutar este notebook desde la carpeta raiz del repositorio.
- Tener instaladas las dependencias indicadas en el `README.md`.

In [24]:
##  %pip install pyspark

from pyspark import SparkContext
try:
    sc.stop()
except:
    pass

In [25]:
import os
from pyspark import SparkConf, SparkContext
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

conf = (SparkConf()
        .setMaster("local[*]")        # Corre Spark en tu máquina local, usando todos los núcleos disponibles del CPU ("local" a secas, corre Spark en modo local con un solo hilo (1 CPU))
        .setAppName("Grupo 11 - TP Etapa 3")
        .set("spark.driver.bindAddress", "127.0.0.1")
        .set("spark.driver.host", "127.0.0.1"))
sc = SparkContext(conf = conf)
sc

<SparkContext master=local[*] appName=Grupo 11 - TP Etapa 3>

In [ ]:
import csv
def load_csv_as_dict_rdd(path):
    raw_rdd = sc.textFile(str(path))
    header = raw_rdd.first()
    headers = next(csv.reader([header]))

    def parse_line(line):
        values = next(csv.reader([line]))
        return dict(zip(headers, values))

    return raw_rdd.filter(lambda line: line != header).map(parse_line)

In [27]:
from pathlib import Path
dataPath = Path("../data")

ingrediente_rdd = load_csv_as_dict_rdd(dataPath / "ingrediente.csv")
relleno_ingrediente_rdd = load_csv_as_dict_rdd(dataPath / "relleno_ingrediente.csv")
cliente_rdd = load_csv_as_dict_rdd(dataPath / "cliente.csv")
compra_rdd = load_csv_as_dict_rdd(dataPath / "compra.csv")
detalle_compra_rdd = load_csv_as_dict_rdd(dataPath / "detalle_compra.csv")
pasta_rdd = load_csv_as_dict_rdd(dataPath / "pasta.csv")


In [28]:
def print_rows(rows, title=None):
    if title:
        print(title)
    for row in rows:
        print(row)

## Procesamiento 1
### ¿Qué pasta fue comprada en la mayor cantidad de franquicias?

**Fase de Map.**
1. Desde `detalle_compra.csv`, cada fila se transforma en `((codigo_pasta, sello), 1)` para representar la compra de una pasta en una franquicia específica.
2. Luego se aplica `distinct()` para conservar una sola ocurrencia por combinación pasta-franquicia.
3. Desde `pasta.csv`, cada fila se transforma en `((codigo_pasta, sello), nombre_pasta)` para asociar cada código con su nombre comercial.

**Fase de Reduce.**
1. Se realiza un `join()` entre ambas estructuras para recuperar el nombre de cada pasta.
2. Luego se remapea cada registro como `(nombre_pasta, 1)`.
3. Se aplica `reduceByKey()` para contar en cuántas franquicias distintas fue comprada cada pasta.
4. Finalmente, se obtiene el valor máximo de franquicias alcanzado y se filtran las pastas que cumplen esa condición.

**Resultado final.** Se devuelve la pasta, o las pastas en caso de empate, que fueron compradas en la mayor cantidad de franquicias.

In [49]:
nombre_por_pasta_y_sello = (
    pasta_rdd
    .map(lambda row: ((row["codigo_pasta"], row["sello"]), row["nombre"]))  # relaciona cada pasta de cada franquicia con su nombre
)

pastas_por_cantidad_de_franquicias = (
    detalle_compra_rdd
    .map(lambda row: ((row["codigo_pasta"], row["sello"]), 1))  # identifica la combinación pasta-franquicia en cada compra
    .distinct()  # deja una sola vez cada pasta por franquicia
    .join(nombre_por_pasta_y_sello)  # recupera el nombre de la pasta
    .map(lambda item: (item[1][1], 1))  # deja como clave el nombre de la pasta y como valor 1
    .reduceByKey(lambda a, b: a + b)  # cuenta en cuántas franquicias fue comprada cada pasta
)

max_franquicias = (
    pastas_por_cantidad_de_franquicias
    .map(lambda item: item[1])  # se queda solo con la cantidad de franquicias
    .max()  # obtiene la cantidad máxima de franquicias alcanzada por alguna pasta
)

pastas_mas_vendidas_en_franquicias = (
    pastas_por_cantidad_de_franquicias
    .filter(lambda item: item[1] == max_franquicias)  # conserva solo las pastas que alcanzan el máximo
    .collect()  # trae el resultado final al driver
)

print_rows(pastas_mas_vendidas_en_franquicias, "Pasta comprada en la mayor cantidad de franquicias:")

Pasta comprada en la mayor cantidad de franquicias:
('Panzotti', 29)


## Procesamiento 2
### ¿Cuál es el gasto promedio por compra en cada franquicia?

**Fase de Map.**
1. En `detalle_compra.csv`, cada linea se transforma en `(id_compra, subtotal)`, donde `subtotal = cantidad_kg * precio_unitario`.
2. En `compra.csv`, cada compra se representa como `(id_compra, id_cliente)`.
3. En `cliente.csv`, cada cliente se representa como `(id_cliente, sello)`.

**Fase de Reduce.**
1. Se suman los subtotales por `id_compra` para obtener el monto total de cada compra.
2. Mediante joins, ese total se vincula primero con el cliente y luego con la franquicia correspondiente.
3. Ya con la clave `sello`, se agrupan los datos como `(monto_total_acumulado, cantidad_de_compras)`.
4. Finalmente, para cada franquicia se calcula el promedio dividiendo el monto acumulado por la cantidad de compras.

**Resultado final.** Se ordenan los promedios de menor a mayor y se usa `collect()` para traer el resultado final y mostrarlo en pantalla.


In [ ]:
subtotal_por_compra = (
    detalle_compra_rdd
    .map(lambda row: (row["id_compra"], float(row["cantidad_kg"]) * float(row["precio_unitario"])))  # calcula el subtotal de cada detalle de compra
    .reduceByKey(lambda a, b: a + b)  # suma los subtotales para obtener el total de cada compra
)

cliente_por_compra = (
    compra_rdd
    .map(lambda row: (row["id_compra"], row["id_cliente"]))  # relaciona cada compra con su cliente
)

sello_por_cliente = (
    cliente_rdd
    .map(lambda row: (row["id_cliente"], row["sello"]))  # relaciona cada cliente con su franquicia/sello
)

promedio_gasto_por_franquicia = (
    subtotal_por_compra
    .join(cliente_por_compra)  # une el total de la compra con el cliente que la realiza
    .map(lambda item: (item[1][1], item[1][0]))  # reordena para dejar como clave el id_cliente y como valor el subtotal de la compra
    .join(sello_por_cliente)  # une cada cliente con su franquicia
    .map(lambda item: (item[1][1], (item[1][0], 1)))  # deja como clave la franquicia y como valor (monto_compra, 1) para luego calcular promedio
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))  # suma montos y cantidad de compras por franquicia
    .map(lambda item: (item[0], round(item[1][0] / item[1][1], 2)))  # calcula el promedio de gasto por compra en cada franquicia
    .takeOrdered(10, key=lambda item: (item[1], item[0]))  # ordena de menor a mayor
)


print_rows(promedio_gasto_por_franquicia, "Promedio de gasto por compra en cada franquicia:")


Promedio de gasto por compra en cada franquicia:
('FR-012', 25765.06)
('FR-005', 25862.58)
('FR-013', 26027.63)
('FR-015', 26177.24)
('FR-003', 26304.69)
('FR-002', 26659.22)
('FR-014', 28283.13)
('FR-001', 28596.18)
('FR-010', 28948.29)
('FR-006', 29226.78)
('FR-007', 30025.35)
('FR-008', 30044.1)
('FR-011', 31182.03)
('FR-009', 32043.34)
('FR-004', 33392.35)


## Procesamiento 3
### ¿Cuáles son las 10 pastas más vendidas por kilos?

**Fase de Map.**
1. Desde `detalle_compra.csv` se genera `((sello, codigo_pasta), cantidad_kg)` para registrar cuántos kilos se vendieron de cada pasta dentro de cada franquicia.
2. Desde `pasta.csv` se genera `((sello, codigo_pasta), nombre_pasta)` para poder traducir luego el codigo al nombre comercial.

**Fase de Reduce.**
1. Se suman los kilos por la clave compuesta `(sello, codigo_pasta)`.
2. Se hace un join con el RDD de pastas para recuperar el nombre.
3. Como una misma pasta puede existir en más de una franquicia, se vuelve a agrupar por `nombre_pasta` y se suman los kilos totales.

**Resultado final.** Se ordena de mayor a menor segun la cantidad de kilos vendidos y se usa `take(10)` para obtener el ranking final.


In [44]:
kg_por_pasta_y_sello = (
    detalle_compra_rdd
    .map(lambda row: ((row["sello"], row["codigo_pasta"]), float(row["cantidad_kg"])))  # toma cada venta y deja como clave (sello, pasta) y como valor los kg vendidos
    .reduceByKey(lambda a, b: a + b)  # suma los kilos vendidos por cada pasta dentro de cada sello
)

nombre_por_pasta_y_sello = (
    pasta_rdd
    .map(lambda row: ((row["sello"], row["codigo_pasta"]), row["nombre"]))  # relaciona cada pasta de cada sello con su nombre
)

top_pastas_por_kg = (
    kg_por_pasta_y_sello
    .join(nombre_por_pasta_y_sello)  # une los kilos vendidos con el nombre de la pasta
    .map(lambda item: (item[1][1], item[1][0]))  # deja como clave el nombre de la pasta y como valor los kg vendidos
    .reduceByKey(lambda a, b: a + b)  # suma los kilos si una misma pasta aparece en mas de un sello
    .takeOrdered(10, key=lambda item: (-item[1], item[0]))  # ordena de mayor a menor
)

print_rows(top_pastas_por_kg, "Top 10 pastas por kilos vendidos:")


Top 10 pastas por kilos vendidos:
('Panzotti', 14392.937)
('Capeletti', 12756.074)
('Ravioloni', 11157.025999999998)
('Ravioles', 10305.064000000002)
('Fusilli', 9764.783)
('Ñoquis', 8372.837)
('Penne', 8349.024000000001)
('Tortellini', 8101.1630000000005)
('Mezzelune', 7940.683)
('Cintas', 7851.788000000001)


## Procesamiento 4
### ¿Qué meses concentran la mayor cantidad de compras?

**Fase de Map.** A partir de `compra.csv`, cada compra se transforma en el par `(MM, 1)`, tomando de `fecha_hora` solamente el mes.

**Fase de Reduce.** Se aplica `reduceByKey()` para sumar la cantidad de compras de cada mes, independiente del año.

**Resultado final.** Se ordenan los meses en forma descendente según la cantidad de compras y se imprime el ranking mensual.”


In [47]:
compras_por_mes = (
    compra_rdd
    .map(lambda row: (row["fecha_hora"][5:7], 1))   # extrae solo el mes (MM) de la fecha_hora (AAAA-MM-DD HH:HH:HH) 
    .reduceByKey(lambda a, b: a + b)  # cuenta cuántas compras hubo en cada mes
    .sortBy(lambda item: (-item[1], item[0]))  # ordena de mayor a menor cantidad de compras
)

top_meses = compras_por_mes.collect()
print_rows(top_meses, "Meses con mayor cantidad de compras:")


Meses con mayor cantidad de compras:
('05', 2446)
('12', 2444)
('01', 2430)
('11', 2410)
('03', 2390)
('06', 2389)
('04', 2386)
('08', 2378)
('07', 2367)
('09', 2341)
('10', 2301)
('02', 2214)
